# CopyLane 판정 인코더 — 실험 카드 3 · 4 : 거짓_과장 칸의 손실만 바꾼다 (박수진, 2026-10-07)

> Baseline = v10 (`copylane-encoder-kcbert-v10-합성O-이유O`) · 골든 `529c970556d7` · 공통 dev 749행 (지문 `4ebfcb231da703b5`)
> 규약 `docs/psj/notes/인코더_개선실험_규약_20261007.md` · 카드 원문 `docs/psj/reports/v10_문턱규칙_거짓과장_실험기록_20261007.md` (커밋 `6d0f929` — **돌리기 전에 올렸다**)
> 🔴 **dev 만 본다. 이 노트북에는 test 채점 셀이 없다** (D-175).

## 무엇을 하나

Baseline 은 유형 있는 위반 510행 중 60행에서 「위반인 줄은 알지만 정답 유형이 후보에 없다」(F1 유형 오인). 그중 32행은 후보가 거짓_과장 하나뿐이다.
후보 규칙(카드 2)과 문턱(카드 5)으로는 값을 치르지 않고 풀리지 않았다. 여기서는 **학습**을 바꾼다 — 학습 행 · 설정은 Baseline 과 같고, **거짓_과장 칸의 손실 가중치 하나만** 다르다.

| 실험 | Baseline 과 다른 점 (하나) | 겨냥한 지표와 목표 (돌리기 전에 적음) |
|---|---|---|
| **카드 3** `카드3-거짓과장누름` | 정답에 다른 유형이 있고 거짓_과장이 없는 학습 행에서, 거짓_과장 칸의 손실 × 3 | ① 유형 오인 60 → **46행 이하** ② 다른 유형 위반 행에 선 거짓_과장 후보 87 → **60행 이하** |
| **카드 4** `카드4-이유거짓끔` | 의결서 「이유」 구역 학습 행에서, 거짓_과장 칸의 손실을 끈다 (다른 유형 라벨은 그대로 학습) | ① 광고 문구 줄의 유형 오인 40 → **30행 이하** ② 광고 문구 줄의 6종 PR-AUC 0.824 유지 (**−0.01 이내**) |

두 카드는 서로 독립이다. 하나씩 Baseline 과 견준다. 잃어도 되는 것은 둘 다 통과 기준 네 개(B · G1 · G2 · G3) 그대로다.

## 판정은 어디서 하나

- 이 노트북의 표(섹션 7)는 **미리 보는 값**이다. Baseline 칸은 규약에 적힌 값을 그대로 넣었다(다시 학습하지 않는다).
- **판정은 공통 채점 스크립트로 한다** — 시드별 dev 행별 확률 CSV 를 받아 `score_encoder_dev.py` 에 넣는다 (섹션 8).

## 실행 순서

1. 런타임 유형 **GPU(T4)** → 위에서부터 끝까지 실행. `golden.jsonl` 을 올린다.
2. 학습은 2 실험 × 시드 3 = **6번** (T4 에서 약 35 ~ 40분). 시드 하나가 끝날 때마다 Drive `copylane_v10_cards/` 에 기록한다 — 런타임이 끊겨도 다시 실행하면 **끝난 시드는 건너뛴다**.
3. 실험마다 **시드별 dev 행별 확률 CSV 3개 · 문턱 zip · 모델 zip** 이 Drive 에 남는다 (**저장소에는 올리지 않는다** · D-249).

## 1. 환경 설정 + 파일 업로드

`sympy` 충돌로 `from transformers import ...` 가 깨지면 런타임 > 세션 다시 시작 후 첫 셀은 건너뛰고 다음 셀부터 실행한다.

In [ ]:
!pip install -q -U sympy transformers accelerate

In [ ]:
import os, json, hashlib, shutil, gc, math, random, csv, time, zipfile
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from IPython.display import display

from google.colab import files as colab_files
if not os.path.exists("golden.jsonl"):
    print("① golden.jsonl 선택")
    colab_files.upload()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

GOLDEN_SHA = sha256("golden.jsonl")
print("golden.jsonl sha256:", GOLDEN_SHA[:16], "…")
# D-176: 입력 지문을 **대조**한다
EXPECTED_GOLDEN_SHA = "529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe"   # 재동결 10-05 (8,383행)
EXPECTED_ROWS = (8383, 5894, 2489)   # 전체 · 학습 · 평가
if GOLDEN_SHA != EXPECTED_GOLDEN_SHA:
    raise SystemExit(f"🔴 golden sha 가 원장과 다르다 — 받은 {GOLDEN_SHA[:16]} · 원장 {EXPECTED_GOLDEN_SHA[:16]} (D-176). "
                     "런타임에 예전 파일이 남았으면 런타임을 새로 시작한다")

## 2. 스위치 — 바꿀 것은 이 셀뿐이다

| 스위치 | 기본값 | 뜻 |
|---|---|---|
| `RUNS` | 카드 3 · 카드 4 | 돌릴 실험 |
| `RETRAIN_BASELINE` | `False` | `True` = Baseline(v10)도 이 노트북의 코드로 다시 학습한다(가중치 전부 1 · 약 20분 더). 코드가 v10 과 같은 결과를 내는지 볼 때만 켠다 |
| `SAVE_MODEL_ZIP` | `True` | 실험마다 가장 좋은 시드의 모델 zip(약 420MB)을 Drive 에 둔다. 채택 후보가 아니면 지워도 된다 |
| `SMOKE` | `False` | `True` = 1분짜리 점검(시드 1 · 1 epoch · 학습 300줄). 수치는 의미 없다 |
| `GATE` · `TARGET` | 아래 값 | 🔒 **규약 · 실험 카드에 적은 값이다. 고치지 않는다** |

In [ ]:
RUN_DEFS = {                      # 이름 → 거짓_과장 칸의 손실 가중치를 어떻게 바꾸나 (그 밖은 Baseline 과 같다)
    "Baseline재현":        {"mode": "none"},                        # 가중치 전부 1 — v10 과 같은 학습
    "카드3-거짓과장누름":  {"mode": "other_up", "factor": 3.0},     # 다른 유형 위반 행(거짓_과장 없음)의 거짓_과장 칸 × 3
    "카드4-이유거짓끔":    {"mode": "reason_off"},                  # 「이유」 구역 행의 거짓_과장 칸 × 0
}
RUNS = ["카드3-거짓과장누름", "카드4-이유거짓끔"]
RETRAIN_BASELINE = False
SAVE_MODEL_ZIP = True
SMOKE = False

# ── 학습 설정 — Baseline(v10)과 같다. 바꾸지 않는다
MODEL_NAME = "beomi/kcbert-base"
SEEDS = [42, 43, 44]
MAX_EPOCHS, PATIENCE = 6, 2
BATCH, LR, WARMUP = 32, 2e-5, 0.1
WEIGHT_DECAY, MAX_LEN = 0.01, 128
FOCAL_GAMMA, POS_WEIGHT_CAP = 1.5, 5
USE_WORD_ROWS = False             # 사례집 낱말 행 제외 (Baseline 과 같다)
DEV_FRAC, DEV_SEED = 0.15, 42     # 공통 dev — 바꾸면 dev 행이 달라진다
TH_OFF = 1.01                     # 편입 대기 칸(기능성화장품_오인)의 문턱 — 꺼 둠 (D-321)
SCORE_VIEW = "v11"                # 회차 · 시드 · 문턱은 dev 749행 전부로 고른다 (규약 4절)

# ── 🔒 통과 기준 (규약 5절) — Baseline 대비
GATE = {
    "prauc_drop": 0.02,           # B  : 6종 dev macro PR-AUC(시드 평균) 하락 허용폭
    "recall_drop": 0.02,          # G1 : 위반 탐지율(유형 있는 위반에 후보가 하나라도) 하락 허용폭
    "type_recall_drop": 0.05,     # G2 : 유형별 재현율 하락 허용폭 (dev 양성 30 이상인 유형만)
    "neg_flag_up": 1,             # G3 : 음성(D · L) 오판정이 늘어도 되는 행 수
    "adopt_prauc": 0.008,         # 채택 후보 ① : PR-AUC 시드 평균이 이만큼 이상 오른다
}
# ── 🔒 겨냥한 지표와 목표 (실험 카드 · 커밋 6d0f929) — 채택 후보 ② : 통과 기준에 걸리지 않고 목표를 전부 채운다
TARGET = {
    "카드3-거짓과장누름": {"mis_max": 46, "false_on_other_max": 60},        # 유형 오인 · 다른 유형 위반 행에 선 거짓_과장 후보
    "카드4-이유거짓끔":   {"ad_mis_max": 30, "ad_prauc_drop": 0.01},        # 광고 문구 줄의 유형 오인 · 광고 문구 줄의 6종 PR-AUC 하락
}
# ── Baseline(v10)의 dev 값 — 규약 1절 · 공통 채점 스크립트로 잰 값. 다시 재지 않고 견줄 때만 쓴다
BASELINE = {
    "seed_prauc": [0.7167, 0.7081, 0.7165], "det": 491, "scored": 510, "hit": 431, "mis": 60, "miss": 19, "neg": 17, "neg_n": 86,
    "type_tp": {"질병_예방치료_표방": (25, 30), "건강기능식품_오인": (33, 42), "의약품_오인": (19, 37), "거짓_과장": (293, 315), "소비자_기만": (62, 98)},
    "false_cand": 520, "false_on_other": 87, "mis_only_false": 32, "ad_mis": 40, "ad_scored": 253, "ad_prauc6": 0.8243, "prauc6": 0.7167,
}

if RETRAIN_BASELINE and "Baseline재현" not in RUNS:
    RUNS = ["Baseline재현"] + RUNS
assert RUNS and all(r in RUN_DEFS for r in RUNS), RUNS
if SMOKE:
    SEEDS, MAX_EPOCHS = [42], 1
NOTEBOOK = "v10-cards"
def run_tag(name):
    return f"copylane-encoder-kcbert-v10-{name}"

OUT_ROOT = "/content/drive/MyDrive/copylane_v10_cards" + ("_smoke" if SMOKE else "")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(OUT_ROOT, exist_ok=True)
except Exception as e:
    OUT_ROOT = "./copylane_v10_cards_out"
    os.makedirs(OUT_ROOT, exist_ok=True)
    print("🟡 Drive 연결 안 됨 — 결과를 런타임에만 둔다(런타임이 끊기면 사라진다):", e)
print("실험:", RUNS, "· 시드", SEEDS, "· 결과 폴더", OUT_ROOT)

## 3. 데이터 + dev 분리 — 공통 dev (Baseline 노트북과 같은 셀)

1. 묶음 = 같은 원문에서 나온 행 (`inj:T1:hf:0:0` → `hf:0:0` · 그 밖은 `id` 의 `#` 앞).
2. 원천마다 묶음의 15% 를 dev 로 (시드 42 · 묶음 이름순 → 섞기).
3. 6종마다 dev 양성이 10건보다 적으면 남은 묶음에서 채운다.
4. dev 행 = dev 묶음의 **문장 단위 · 합성 아님** 행.

같은 골든이면 **749줄 · 지문 `4ebfcb231da703b5`** 가 나와야 한다 — v11 의 `dev_ids_v11_seed42.csv` 와 같은 행 · 같은 순서다. 다르면 멈춘다.

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

SEL = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]   # 모델을 고르는 6종 (D-321 결정 4)
NEW_CONF = ["부당_비교광고", "비방광고"]
CONF8 = SEL + NEW_CONF                       # 확정 8종 — 문턱을 dev 에서 고른다
WAITING = ["기능성화장품_오인"]              # 편입 대기 — 칸은 두고 문턱은 끈다
LABEL_LIST = CONF8 + WAITING                 # 저장 순서 (팀장 10-06)
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
N_SEL, N8, N_ALL = len(SEL), len(CONF8), len(LABEL_LIST)

def norm_text(s):
    return " ".join(s.split())

def group_key(rid):
    if rid.startswith("inj:"):               # 'inj:T1:hf:0:0' → 원본 승인 문구 'hf:0:0'
        return rid.split(":", 2)[2]
    return rid.split("#")[0]

def normalize(r):
    cond, labels = r.get("조건"), list(r.get("labels") or [])
    return {"id": r["id"], "group": group_key(r["id"]), "text": r["text"], "labels": labels,
            # 조건 M · D · L 은 라벨이 적혀 있어도 양성이 아니다 (D-296 개정 2) — target 이 학습 · 채점의 정답이다
            "target": [] if cond in ("M", "D", "L") else labels,
            "split": r["split"], "provenance": r.get("provenance"), "origin": r.get("origin"),
            "zone": r.get("구역"), "cond": cond, "unit": r.get("unit")}

def is_reason(r):
    return r["provenance"] == "ftc_decisions_body" and r["zone"] == "이유"

def is_inj(r):
    return r["origin"] == "injected" or r["id"].startswith("inj:")

def bucket(r):
    # 8종 기준 (run_judge_dist.py 와 같은 갈래)
    if r["cond"] == "M": return "M"
    if r["cond"] == "D": return "D"
    if r["cond"] == "L": return "L"
    if any(l in CONF8 for l in r["labels"]): return "scored"
    if r["labels"]: return "대기"
    return "pending" if r["cond"] in ("C", "A", "B") else "neg"

golden = load_jsonl("golden.jsonl")
all_rows = [normalize(r) for r in golden]
unknown = {l for r in all_rows for l in r["labels"]} - set(LABEL_LIST)
assert not unknown, f"🔴 예상 밖 라벨: {unknown}"
train_all = [r for r in all_rows if r["split"] == "train"]
_got = (len(all_rows), len(train_all), len(all_rows) - len(train_all))
if _got != EXPECTED_ROWS:
    raise SystemExit(f"🔴 행 수가 다르다 — 받은 {_got} · 기대 {EXPECTED_ROWS} (D-176)")
del golden, all_rows      # 🔴 여기서부터 test 행은 메모리에도 없다 — 이 노트북은 train(→ train · dev)만 본다

# ══ dev 분리 — v11 (encoder_finetune_colab_v11_kcbert_1006) 섹션 3 과 같은 코드
def dev_eligible(r):
    return r["unit"] == "문장" and r["origin"] != "injected"

groups = defaultdict(list)
for r in train_all:
    groups[r["group"]].append(r)

prov_groups = defaultdict(list)             # 원천별 15% (묶음 단위 · 시드 고정)
for g in sorted(groups):
    elig = [r for r in groups[g] if dev_eligible(r)]
    if elig:
        prov_groups[Counter(r["provenance"] for r in elig).most_common(1)[0][0]].append(g)
rng = random.Random(DEV_SEED)
dev_groups = set()
for prov in sorted(prov_groups):
    gs = prov_groups[prov][:]
    rng.shuffle(gs)
    dev_groups.update(gs[:max(1, math.ceil(DEV_FRAC * len(gs)))])

def conf_pos(rows, l):
    return sum(1 for r in rows if dev_eligible(r) and l in r["target"])
rest = [g for g in sorted(groups) if g not in dev_groups]
rng.shuffle(rest)
for l in SEL:                               # 6종마다 dev 양성이 10건보다 적으면 채운다
    total = conf_pos(train_all, l)
    want = min(10, math.ceil(DEV_FRAC * total))
    have = sum(conf_pos(groups[g], l) for g in dev_groups)
    for g in rest:
        if have >= want:
            break
        k = conf_pos(groups[g], l)
        if k and g not in dev_groups:
            dev_groups.add(g); have += k

dev_rows = [r for g in sorted(dev_groups) for r in groups[g] if dev_eligible(r)]
dev_texts = {norm_text(r["text"]) for r in dev_rows}
DEV_SHA = hashlib.sha256("|".join(sorted(r["id"] for r in dev_rows)).encode()).hexdigest()[:16]
DEV_EXPECT = {"529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe": (749, "4ebfcb231da703b5")}   # 골든 sha → v11 dev (줄 수 · 지문)
if GOLDEN_SHA in DEV_EXPECT:
    if (len(dev_rows), DEV_SHA) != DEV_EXPECT[GOLDEN_SHA]:
        raise SystemExit(f"🔴 dev 가 v11 과 다르다 — {len(dev_rows)}줄 · 지문 {DEV_SHA} · 기대 {DEV_EXPECT[GOLDEN_SHA]}. 분리 코드를 건드렸는지 본다")
    print(f"✅ dev = v11 과 같은 행: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
else:
    print(f"🟡 이 골든의 v11 dev 지문을 모른다 — 대조 없이 간다: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
assert len({r["id"] for r in dev_rows}) == len(dev_rows), "dev 에 같은 id 가 둘 이상이다"

# ── dev 행의 갈래 · 채점 방식
for r in dev_rows:
    r["bucket"] = bucket(r)
B = np.array([r["bucket"] for r in dev_rows])
IS_REASON = np.array([is_reason(r) for r in dev_rows])
VIEW = {"v11": np.ones(len(dev_rows), dtype=bool),
        "A": ~IS_REASON & ~np.isin(B, ["M", "pending"])}
SEL_MASK = VIEW[SCORE_VIEW]
BUCKETS = ["scored", "대기", "pending", "M", "D", "L", "neg"]
BUCKET_KO = {"scored": "유형 있는 위반", "대기": "편입 대기 유형만", "pending": "유형 없는 위반", "M": "조건 M", "D": "조건 D", "L": "조건 L(적법)", "neg": "그 밖 음성"}
print("\ndev 갈래 (전체 / 그중 「이유」 구역):")
for b in BUCKETS:
    if (B == b).any():
        print(f"  {BUCKET_KO[b]:12s} {int((B == b).sum()):4d} / {int(((B == b) & IS_REASON).sum()):4d}")
print(f"  합계           {len(dev_rows):4d} / {int(IS_REASON.sum()):4d}")
print(f"채점 행 — v11 방식 {int(VIEW['v11'].sum())}줄 · A 방식 {int(VIEW['A'].sum())}줄  → 이번 선택 방식: {SCORE_VIEW}")

## 4. 학습셋 · 손실 가중치

**학습 행은 Baseline 과 같다** (3,964행). dev 묶음을 뺀 train 에서 순서대로:
dev 와 같은 문구 제거 → 조건 M 제외 → 유형 없는 위반 제외 → 낱말 행 제외 → 같은 문구 병합. 합성 문구 · 「이유」 구역은 넣는다. D · L 은 음성이다.

실험이 바꾸는 것은 **행 × 유형 칸마다 붙는 손실 가중치**뿐이다. 기본은 전부 1 이고, 거짓_과장 칸만 다음처럼 바뀐다.

| 실험 | 가중치가 바뀌는 행 | 거짓_과장 칸의 가중치 | 그 행의 다른 칸 |
|---|---|---|---|
| 카드 3 | 정답에 다른 유형이 있고 거짓_과장이 없는 행 (1,593행) | 3 | 그대로 1 |
| 카드 4 | 의결서 「이유」 구역 행 (1,417행 · 그중 거짓_과장 양성 1,151) | 0 | 그대로 1 |

- 손실은 가중치를 곱한 뒤 **Baseline 과 같은 수(행 × 9칸)로 나눈다** — 다른 칸이 받는 기울기는 Baseline 과 같다.
- `pos_weight` 는 Baseline 과 같은 값(학습 행 전체로 계산)을 쓴다. 한 가지만 바꾼다.
- 카드 3 에서 유형이 없는 음성 행(D · L · 522행)은 건드리지 않는다. 「다른 유형의 위반」에서만 거짓_과장을 더 세게 누른다.

In [ ]:
base_pool = [r for r in train_all if r["group"] not in dev_groups]
STEPS = [("dev 묶음 뺀 train", len(base_pool))]
pool0 = [r for r in base_pool if norm_text(r["text"]) not in dev_texts];                 STEPS.append(("dev 와 같은 문구 제거", len(pool0)))
pool0 = [r for r in pool0 if r["cond"] != "M"];                                          STEPS.append(("조건 M 제외", len(pool0)))
pool0 = [r for r in pool0 if not (r["cond"] in ("C", "A", "B") and not r["labels"])];    STEPS.append(("유형 없는 위반 제외", len(pool0)))
if not USE_WORD_ROWS:
    pool0 = [r for r in pool0 if r["unit"] != "낱말"];                                   STEPS.append(("낱말 행 제외", len(pool0)))
print(" → ".join(f"{k} {n:,}" for k, n in STEPS))
assert not any(r["cond"] == "M" for r in pool0) and not any(r["target"] for r in pool0 if r["cond"] in ("D", "L"))

merged = {}
for r in pool0:                                      # 같은 문구 병합 — 라벨 합집합 · 이유/비이유는 따로 (Baseline 과 같은 방식)
    k = (norm_text(r["text"]), is_reason(r))
    if k in merged:
        merged[k]["target"] = sorted(set(merged[k]["target"]) | set(r["target"]))
    else:
        merged[k] = dict(r, target=list(r["target"]))
TRAIN_ROWS = list(merged.values())
if SMOKE:
    random.Random(0).shuffle(TRAIN_ROWS); TRAIN_ROWS = TRAIN_ROWS[:300]
assert not (set(r["id"] for r in TRAIN_ROWS) & set(r["id"] for r in dev_rows)), "학습 행에 dev 행이 섞였다"
assert not ({norm_text(r["text"]) for r in TRAIN_ROWS} & dev_texts), "학습 행에 dev 와 같은 문구가 있다"

FALSE = "거짓_과장"
F_IDX = LABEL2ID[FALSE]
def loss_weights(rows, cfg):
    """행 × 9칸 손실 가중치. 거짓_과장 칸만 바뀐다."""
    W = np.ones((len(rows), N_ALL), dtype=np.float32)
    if cfg["mode"] == "other_up":
        hit = np.array([bool(r["target"]) and FALSE not in r["target"] for r in rows])
        W[hit, F_IDX] = cfg["factor"]
    elif cfg["mode"] == "reason_off":
        hit = np.array([is_reason(r) for r in rows])
        W[hit, F_IDX] = 0.0
    else:
        assert cfg["mode"] == "none", cfg
        hit = np.zeros(len(rows), dtype=bool)
    return W, hit

LOSS_W, TRAIN_STATS = {}, {}
_pos = np.array([FALSE in r["target"] for r in TRAIN_ROWS])
for name in RUN_DEFS:
    W, hit = loss_weights(TRAIN_ROWS, RUN_DEFS[name])
    assert (np.delete(W, F_IDX, axis=1) == 1).all(), "거짓_과장이 아닌 칸의 가중치가 바뀌었다"
    LOSS_W[name] = W
    TRAIN_STATS[name] = {"학습 행": len(TRAIN_ROWS), "합성": sum(map(is_inj, TRAIN_ROWS)), "이유 구역": sum(map(is_reason, TRAIN_ROWS)),
                         "가중치가 바뀐 행": int(hit.sum()), "그중 거짓_과장 양성": int((hit & _pos).sum()), "그중 「이유」 구역": int(sum(is_reason(r) for r, h in zip(TRAIN_ROWS, hit) if h)),
                         "거짓_과장 칸 가중치": "1" if not hit.any() else f"{W[hit, F_IDX][0]:g}",
                         "거짓_과장을 배우는 양성 행": int((_pos & (W[:, F_IDX] > 0)).sum()), "음성 행": int((~_pos & (W[:, F_IDX] > 0)).sum())}
print("\n■ 학습셋 · 손실 가중치" + ("   (SMOKE — 300줄로 줄였다)" if SMOKE else ""))
display(pd.DataFrame(TRAIN_STATS).T)
EXPECT_TRAIN = {"학습 행": 3964, "카드3-거짓과장누름": 1593, "카드4-이유거짓끔": 1417}     # 재동결 10-05 판 · 로컬 대조 10-07
if GOLDEN_SHA in DEV_EXPECT and not SMOKE and USE_WORD_ROWS is False:
    _got = {"학습 행": len(TRAIN_ROWS), **{n: TRAIN_STATS[n]["가중치가 바뀐 행"] for n in ("카드3-거짓과장누름", "카드4-이유거짓끔")}}
    if _got != EXPECT_TRAIN:
        raise SystemExit(f"🔴 학습 행 · 가중치가 바뀐 행 수가 기대와 다르다 — 받은 {_got} · 기대 {EXPECT_TRAIN}. 학습셋 코드를 건드렸는지 본다")
    print("✅ 학습 3,964행 (Baseline 과 같다) · 카드 3 은 1,593행 · 카드 4 는 1,417행의 거짓_과장 칸이 바뀐다")

## 5. 손실 · 지표

- Focal Loss(γ 1.5) · α = pos_weight = √(음성/양성) 을 1 ~ 5 로 자른 값 — Baseline 과 같다.
- **달라진 곳은 한 줄이다** — 칸마다의 손실에 섹션 4 의 가중치를 곱한다. 가중치가 전부 1 이면 Baseline 과 같은 식이다.
- 회차(조기 종료) · 시드 = **6종 dev macro PR-AUC** 가 가장 높은 것. 문턱 = 확정 8종마다 dev F1 최대 (규약 4절).

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          EarlyStoppingCallback, DataCollatorWithPadding, set_seed)
from sklearn.metrics import average_precision_score, f1_score

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)

def multihot(labels):
    v = np.zeros(N_ALL, dtype=np.float32)
    for l in labels:
        v[LABEL2ID[l]] = 1.0
    return v

def to_ds(rows, W=None):
    W = np.ones((len(rows), N_ALL), dtype=np.float32) if W is None else W
    assert W.shape == (len(rows), N_ALL)
    ds = Dataset.from_dict({"text": [r["text"] for r in rows], "labels": [multihot(r["target"]) for r in rows],
                            "loss_w": [w.astype(np.float32) for w in W]})                # 행 × 9칸 손실 가중치 (섹션 4)
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "loss_w"])
    return ds

Y_DEV = np.stack([multihot(r["target"]) for r in dev_rows]).astype(int)
dev_ds = to_ds(dev_rows)

def pos_weight_of(rows):
    Y = np.stack([multihot(r["target"]) for r in rows])
    pos = Y.sum(0); neg = len(Y) - pos
    return np.clip(np.sqrt(neg / np.clip(pos, 1e-6, None)), 1, POS_WEIGHT_CAP)

class FocalTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._pw = torch.tensor(pos_weight, dtype=torch.float32)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float()
        w = inputs.pop("loss_w").float()             # 🔴 Baseline 과 다른 한 줄 — 칸마다의 손실 가중치
        outputs = model(**inputs)
        logits = outputs.logits.float()
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * labels + (1 - p) * (1 - labels)
        loss = (w * (self._pw.to(logits.device) * labels + (1 - labels)) * (1 - p_t) ** FOCAL_GAMMA * bce).mean()
        return (loss, outputs) if return_outputs else loss

def safe(fn, y, p):
    return float(fn(y, p)) if 0 < y.sum() < len(y) else None

def macro_prauc(y, p, n=N_SEL):
    # 앞 n 칸의 PR-AUC 평균 — 양성이 있는 유형만. n=6 이 모델을 고르는 기준이다
    v = [safe(average_precision_score, y[:, i], p[:, i]) for i in range(n)]
    v = [x for x in v if x is not None]
    return float(np.mean(v)) if v else float("nan")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    p = 1 / (1 + np.exp(-logits))
    return {"macro_prauc": macro_prauc(labels[SEL_MASK].astype(int), p[SEL_MASK])}

TH_GRID = np.arange(0.1, 0.9001, 0.025)
def pick_thresholds(y, p):
    th = {}
    for i, l in enumerate(CONF8):
        if y[:, i].sum() < 10:
            th[l] = 0.5
            continue
        fs = [f1_score(y[:, i], p[:, i] >= t, zero_division=0) for t in TH_GRID]
        th[l] = float(round(TH_GRID[int(np.argmax(fs))], 3))
    th[WAITING[0]] = TH_OFF
    return th

def predict_probs(mdl, rows, bs=64):
    mdl.eval()
    dev_ = next(mdl.parameters()).device
    out = []
    for s in range(0, len(rows), bs):
        enc = tokenizer([r["text"] for r in rows[s:s + bs]], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(dev_)
        with torch.no_grad():
            out.append(torch.sigmoid(mdl(**enc).logits.float()).cpu().numpy())
    return np.concatenate(out)

print("dev 양성 (선택 방식", SCORE_VIEW, "):", {l: int(Y_DEV[SEL_MASK, i].sum()) for i, l in enumerate(LABEL_LIST)})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 — CPU 로는 매우 느리다. 런타임 유형을 GPU 로 바꾼다")

## 6. 학습 — 실험 × 시드

🔒 **기준과 목표를 먼저 찍고 학습한다.** 시드 하나가 끝나면 dev 확률을 Drive 에 적는다 — 다시 실행하면 끝난 시드는 건너뛴다(`♻️`).
실험 하나가 끝나면 **시드별 dev 행별 확률 CSV 3개 · 문턱 zip · (가장 좋은 시드의) 모델 zip** 을 Drive 에 둔다.

In [ ]:
print("🔒 통과 기준 (규약 5절) — Baseline(v10) 대비 · 가장 좋은 시드 · 그 시드의 dev 문턱")
print(f"  B   6종 dev macro PR-AUC(시드 평균) 하락이 {GATE['prauc_drop']} 이내   (Baseline {np.mean(BASELINE['seed_prauc']):.4f})")
print(f"  G1  위반 탐지율 하락이 {GATE['recall_drop'] * 100:.0f}%p 이내   (Baseline {BASELINE['det']}/{BASELINE['scored']})")
print(f"  G2  유형별 재현율 하락이 {GATE['type_recall_drop'] * 100:.0f}%p 이내 (dev 양성 30 이상인 유형만)")
print(f"  G3  음성(D · L) 오판정이 +{GATE['neg_flag_up']}행 이내   (Baseline {BASELINE['neg']}/{BASELINE['neg_n']})")
print("🔒 겨냥한 지표와 목표 (실험 카드 · 커밋 6d0f929)")
print(f"  카드 3  ① 유형 오인 {BASELINE['mis']} → {TARGET['카드3-거짓과장누름']['mis_max']}행 이하  ② 다른 유형 위반 행에 선 거짓_과장 후보 {BASELINE['false_on_other']} → {TARGET['카드3-거짓과장누름']['false_on_other_max']}행 이하")
print(f"  카드 4  ① 광고 문구 줄의 유형 오인 {BASELINE['ad_mis']} → {TARGET['카드4-이유거짓끔']['ad_mis_max']}행 이하  ② 광고 문구 줄의 6종 PR-AUC {BASELINE['ad_prauc6']:.3f} 에서 −{TARGET['카드4-이유거짓끔']['ad_prauc_drop']} 이내")
print(f"  채택 후보 = 통과 기준에 걸리지 않고 + (PR-AUC 시드 평균 +{GATE['adopt_prauc']} 이상  또는  그 카드의 목표를 전부 채운다)")
print("  ※ 판정은 공통 채점 스크립트(score_encoder_dev.py)로 한다. 아래 표는 미리 보는 값이다.\n")

CFG_SIG = [NOTEBOOK, GOLDEN_SHA, DEV_SHA, SCORE_VIEW, MODEL_NAME, MAX_EPOCHS, PATIENCE, BATCH, LR, WARMUP, WEIGHT_DECAY, MAX_LEN,
           FOCAL_GAMMA, POS_WEIGHT_CAP, USE_WORD_ROWS, SMOKE]
def sig_of(name, seed):
    ids = hashlib.sha256("|".join(r["id"] for r in TRAIN_ROWS).encode()).hexdigest()
    w = hashlib.sha256(LOSS_W[name].tobytes()).hexdigest()
    return hashlib.sha256(json.dumps(CFG_SIG + [name, RUN_DEFS[name], seed, ids, w], ensure_ascii=False).encode()).hexdigest()[:16]

def train_seed(name, seed, get_ds, pw):
    tag = run_tag(name)
    keep, local = f"{OUT_ROOT}/{tag}/seed{seed}", f"./runs/{tag}/seed{seed}"
    done = f"{keep}/done.json"
    if os.path.exists(done) and os.path.exists(f"{keep}/dev_p.npy"):
        res = json.load(open(done, encoding="utf-8"))
        if res.get("sig") == sig_of(name, seed):
            res["dev_p"] = np.load(f"{keep}/dev_p.npy")
            print(f"♻️ {name} · seed {seed}: 이미 끝남 — 건너뜀 (best epoch {res['best_epoch']})")
            return res
        print(f"{name} · seed {seed}: 저장된 기록이 지금 설정과 다르다 — 다시 학습한다")
    t0 = time.time()
    train_ds = get_ds()
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_ALL, problem_type="multi_label_classification",
        id2label=dict(enumerate(LABEL_LIST)), label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=f"./runs/{tag}/ckpt{seed}", eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64, num_train_epochs=MAX_EPOCHS,
        learning_rate=LR, weight_decay=WEIGHT_DECAY, lr_scheduler_type="linear",
        warmup_steps=int(WARMUP * math.ceil(len(train_ds) / BATCH) * MAX_EPOCHS),
        load_best_model_at_end=True, metric_for_best_model="macro_prauc", greater_is_better=True,
        logging_steps=50, report_to="none", remove_unused_columns=False, seed=seed,
        fp16=torch.cuda.is_available())
    trainer = FocalTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds, data_collator=collator,
                           compute_metrics=compute_metrics, pos_weight=pw,
                           callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()
    hist = [h for h in trainer.state.log_history if "eval_macro_prauc" in h]
    best = max(hist, key=lambda h: h["eval_macro_prauc"])
    dev_p = predict_probs(trainer.model, dev_rows)
    shutil.rmtree(local, ignore_errors=True)
    trainer.save_model(local)
    shutil.rmtree(f"./runs/{tag}/ckpt{seed}", ignore_errors=True)
    res = {"sig": sig_of(name, seed), "seed": seed, "best_epoch": int(round(best["epoch"])), "epochs_run": int(round(hist[-1]["epoch"])),
           "curve": [[int(round(h["epoch"])), round(h["eval_macro_prauc"], 4)] for h in hist], "minutes": round((time.time() - t0) / 60, 1)}
    os.makedirs(keep, exist_ok=True)
    np.save(f"{keep}/dev_p.npy", dev_p)
    shutil.copy(f"{local}/config.json", f"{keep}/config.json")
    json.dump(res, open(done, "w", encoding="utf-8"), ensure_ascii=False)       # 🔴 done.json 을 맨 끝에 쓴다 — 있으면 이 시드는 끝난 것이다
    res["dev_p"] = dev_p
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res

def scheme_of(name, R):
    s, st = R["best_seed"], TRAIN_STATS[name]
    return {
        "label_list": LABEL_LIST, "label_thresholds": R["thresholds"],
        "confirmed_labels": CONF8, "waiting_labels": WAITING, "legacy_confirmed_labels": SEL,
        "waiting_note": f"편입 대기(D-321) — 칸은 두되 문턱 {TH_OFF} 로 꺼 둔다",
        "selection_note": "회차 · 시드는 6종 dev macro PR-AUC 로 고른다 (D-321 결정 4)",
        "backbone": MODEL_NAME, "notebook": f"{NOTEBOOK} (박수진 · 실험 카드 3 · 4 — 거짓_과장 칸의 손실 가중치)", "experiment": run_tag(name),
        "baseline": "copylane-encoder-kcbert-v10-합성O-이유O", "loss_weight_rule": RUN_DEFS[name],
        "loss_weight_rows": st["가중치가 바뀐 행"], "loss_weight_note": "거짓_과장 칸의 손실 가중치만 바꿨다 — other_up = 다른 유형 위반 행(거짓_과장 없음) × factor · reason_off = 의결서 이유 구역 행 × 0 · none = 전부 1",
        "use_injected": True, "use_reason_zone": True, "use_word_rows": USE_WORD_ROWS,
        "seed": s, "best_epoch": R["seeds"][s]["best_epoch"], "golden_sha256": GOLDEN_SHA, "golden_rows": list(EXPECTED_ROWS),
        "train_rows": st["학습 행"], "train_injected_rows": st["합성"], "train_reason_rows": st["이유 구역"],
        "train_label_pos": {l: sum(l in r["target"] for r in TRAIN_ROWS) for l in LABEL_LIST},
        "train_rule": "조건 M 제외 · D · L 음성 · 유형 없는 위반 제외" + ("" if USE_WORD_ROWS else " · 낱말 행 제외"),
        "dev_rule": f"v11 과 같다 — 원천별 묶음 {DEV_FRAC:.0%} + 6종 보충 · 시드 {DEV_SEED} · 문장 단위 · 합성 아님",
        "dev_sha": DEV_SHA, "dev_rows": len(dev_rows), "score_view": SCORE_VIEW, "score_rows": int(SEL_MASK.sum()),
        "score_view_note": "v11 = dev 전부(M · 유형 없는 위반 · D · L 음성) / A = 이유 구역 · M · 유형 없는 위반을 뺀 줄",
        "threshold_note": "가장 좋은 시드의 dev F1 최대 · 격자 0.1~0.9(0.025) · dev 양성 10건 미만이면 0.5. 적용은 확률 >= threshold.",
        "thresholds_by_seed": {str(k): v["thresholds"] for k, v in R["seeds"].items()},
        "dev_macro_prauc6_by_seed": {str(k): v["prauc6"] for k, v in R["seeds"].items()},
        "final_test": False, "test_note": "이 노트북은 test 를 채점하지 않는다 (D-175)", "smoke": SMOKE,
    }

def export_run(name, R):
    """시드별 dev 행별 확률 CSV(규약 3절) · 문턱 zip · (스위치) 가장 좋은 시드의 모델 zip 을 결과 폴더에 둔다."""
    tag, s = run_tag(name), R["best_seed"]
    scheme, out = scheme_of(name, R), []
    for sd in SEEDS:
        csv_path = f"{OUT_ROOT}/{tag}_seed{sd}_dev행별확률.csv"
        with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:      # 문구는 넣지 않는다 (id · 갈래 · 확률만)
            w = csv.writer(f)
            w.writerow(["id", "조건", "bucket", "이유구역"] + [f"p_{l}" for l in LABEL_LIST])
            for r, row in zip(dev_rows, R["seeds"][sd]["dev_p"]):
                w.writerow([r["id"], r["cond"] or "", r["bucket"], int(is_reason(r))] + [f"{x:.6f}" for x in row])
        out.append(os.path.basename(csv_path))
    with zipfile.ZipFile(f"{OUT_ROOT}/{tag}_scheme.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("label_scheme.json", json.dumps(scheme, ensure_ascii=False, indent=2))
        z.write(f"{OUT_ROOT}/{tag}/seed{s}/config.json", "config.json")
    out.append(f"{tag}_scheme.zip")
    local = f"./runs/{tag}/seed{s}"
    if not SAVE_MODEL_ZIP:
        pass
    elif not os.path.exists(f"{local}/config.json"):
        if os.path.exists(f"{OUT_ROOT}/{tag}.zip") and R.get("zip_seed") == s:
            out.append(f"{tag}.zip (앞서 저장한 것)")
        else:
            print(f"  🟡 {name}: 가장 좋은 시드 {s} 의 가중치가 이 런타임에 없다(앞 세션에서 학습) — 모델 zip 은 못 만든다. "
                  f"필요하면 Drive 의 {tag} 폴더를 지우고 이 실험만 다시 돌린다")
    else:
        sdir = f"./export/{tag}"
        shutil.rmtree(sdir, ignore_errors=True)
        m = AutoModelForSequenceClassification.from_pretrained(local)
        assert [m.config.id2label[i] for i in range(m.config.num_labels)] == LABEL_LIST, "저장 칸 순서가 다르다 (D-220)"
        m.save_pretrained(sdir, safe_serialization=True)
        tokenizer.save_pretrained(sdir)
        assert os.path.exists(f"{sdir}/tokenizer.json"), "tokenizer.json 이 없다 (앱 로더 필수 파일)"
        json.dump(scheme, open(f"{sdir}/label_scheme.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        try:
            zp = shutil.make_archive(f"./export/{tag}", "zip", sdir)
            shutil.copy(zp, f"{OUT_ROOT}/{tag}.zip")
            json.dump({"zip_seed": s}, open(f"{OUT_ROOT}/{tag}/zip_seed.json", "w"))
            out.append(f"{tag}.zip")
            os.remove(zp); shutil.rmtree(sdir, ignore_errors=True)          # 런타임 디스크 — 결과 폴더에 옮겼으면 지운다
        except Exception as e:
            print(f"  🟡 {name}: 모델 zip 저장 안 됨 (Drive 용량?) — {e}")
        del m
    return out

RESULTS = {}
T_ALL = time.time()
for name in RUNS:
    tag, rows = run_tag(name), TRAIN_ROWS
    pw = pos_weight_of(rows)                      # Baseline 과 같은 값 — 학습 행 전체로 계산한다
    print(f"\n{'=' * 22} {name} · 학습 {len(rows):,}행 · 거짓_과장 칸 가중치가 바뀐 행 {TRAIN_STATS[name]['가중치가 바뀐 행']:,} {'=' * 22}")
    print("pos_weight:", {l: round(float(pw[i]), 2) for i, l in enumerate(LABEL_LIST)})
    _cache, seeds = {}, {}
    def get_ds():                                 # 학습할 시드가 있을 때만 토크나이즈한다
        if "ds" not in _cache:
            _cache["ds"] = to_ds(rows, LOSS_W[name])
        return _cache["ds"]
    for seed in SEEDS:
        res = train_seed(name, seed, get_ds, pw)
        y, p = Y_DEV[SEL_MASK], res["dev_p"][SEL_MASK]
        res["prauc6"], res["prauc8"] = macro_prauc(y, p, N_SEL), macro_prauc(y, p, N8)
        res["thresholds"] = pick_thresholds(y, p)
        seeds[seed] = res
        print(f"  seed {seed}: 6종 dev macro PR-AUC {res['prauc6']:.4f} (8종 {res['prauc8']:.4f}) · best epoch {res['best_epoch']}/{res['epochs_run']} · 곡선 {res['curve']} · {res['minutes']}분")
    best = max(SEEDS, key=lambda s: seeds[s]["prauc6"])
    _zs = f"{OUT_ROOT}/{tag}/zip_seed.json"
    R = {"seeds": seeds, "best_seed": best, "thresholds": seeds[best]["thresholds"],
         "prauc6_mean": float(np.mean([seeds[s]["prauc6"] for s in SEEDS])), "prauc6_std": float(np.std([seeds[s]["prauc6"] for s in SEEDS])),
         "zip_seed": json.load(open(_zs))["zip_seed"] if os.path.exists(_zs) else None}
    RESULTS[name] = R
    print(f"[{name}] 6종 dev macro PR-AUC {R['prauc6_mean']:.4f} ± {R['prauc6_std']:.4f} · 가장 좋은 시드 {best}")
    print("  문턱:", {l: R["thresholds"][l] for l in CONF8})
    print("  저장:", export_run(name, R))
    for s in SEEDS:                               # 디스크 — 가장 좋은 시드의 가중치만 남긴다
        if s != best:
            shutil.rmtree(f"./runs/{tag}/seed{s}", ignore_errors=True)
print(f"\n학습 끝 — {round((time.time() - T_ALL) / 60, 1)}분")

## 7. 결과 — 미리 보기

- **표 1** 가장 좋은 시드 · 그 시드의 문턱으로 잰 값. Baseline 칸은 규약의 값이다.
- **표 2** 유형별 재현율 · 정밀도.
- **표 3** 통과 기준(B · G1 · G2 · G3)과 그 카드의 목표.

읽는 법
- 「유형 오인」 = 유형 있는 위반에 후보는 섰는데 정답 유형이 후보에 없다. 「놓침」 = 후보가 하나도 없다.
- 「광고 문구 줄」 = 의결서 「이유」 구역 · 조건 M · 유형 없는 위반을 뺀 줄 (325행 · 그중 유형 있는 위반 253행).
- 재현율 기준(G1 · G2)은 문턱이 옮겨 가기만 해도 걸린다. **문턱을 맞춘 비교는 공통 채점 스크립트가 낸다** (섹션 8).

In [ ]:
SC = B == "scored"
NEG = np.isin(B, ["D", "L", "neg"])
AD = ~IS_REASON & ~np.isin(B, ["M", "pending"])          # 광고 문구 줄
Y8 = Y_DEV[:, :N8] == 1

def measure(p, th):
    fl = p[:, :N8] >= np.array([th[l] for l in CONF8])
    any_, hit = fl.any(1), (fl & Y8).any(1)
    mis = any_ & ~hit
    out = {"scored": int(SC.sum()), "det": int(any_[SC].sum()), "hit": int(hit[SC].sum()), "mis": int(mis[SC].sum()),
           "miss": int((~any_)[SC].sum()), "neg": int(any_[NEG].sum()), "neg_n": int(NEG.sum()),
           "D": (int(any_[B == "D"].sum()), int((B == "D").sum())), "L": (int(any_[B == "L"].sum()), int((B == "L").sum())),
           "false_cand": int(fl[:, F_IDX].sum()), "false_tp": int((fl[:, F_IDX] & Y8[:, F_IDX]).sum()),
           "false_on_other": int((fl[:, F_IDX] & ~Y8[:, F_IDX] & SC).sum()),
           "mis_only_false": int((mis & SC & fl[:, F_IDX] & (fl.sum(1) == 1)).sum()),
           "ad_scored": int((SC & AD).sum()), "ad_mis": int((mis & SC & AD).sum()),
           "prauc6": macro_prauc(Y_DEV, p), "prauc8": macro_prauc(Y_DEV, p, N8), "ad_prauc6": macro_prauc(Y_DEV[AD], p[AD]),
           "cand_avg": float(fl[SC & any_].sum(1).mean()) if (SC & any_).any() else float("nan")}
    out["type"] = {l: {"n": int(Y8[:, i].sum()), "tp": int((fl[:, i] & Y8[:, i]).sum()), "pp": int(fl[:, i].sum())} for i, l in enumerate(CONF8)}
    return out

def rate(k, n):
    return "—" if n == 0 else f"{k}/{n} ({k / n * 100:.1f}%)"

M = {}
for name in RUNS:
    R = RESULTS[name]
    M[name] = measure(R["seeds"][R["best_seed"]]["dev_p"], R["thresholds"])
    M[name]["per_seed"] = {s: measure(R["seeds"][s]["dev_p"], R["seeds"][s]["thresholds"]) for s in SEEDS}
b = BASELINE
b_mean, b_std = float(np.mean(b["seed_prauc"])), float(np.std(b["seed_prauc"]))

# ── 표 1
_t = {"Baseline v10 (규약 값)": {
    "6종 PR-AUC 시드 평균": f"{b_mean:.4f} ± {b_std:.4f}", "선택 시드": "42", "선택 시드 PR-AUC (dev 전체 · 광고 문구 줄)": f"{b['prauc6']:.3f} · {b['ad_prauc6']:.3f}",
    "위반 탐지 ↑": rate(b["det"], b["scored"]), "정답 유형 적중 ↑": rate(b["hit"], b["scored"]), "유형 오인 ↓ (거짓_과장 하나만)": f"{rate(b['mis'], b['scored'])} ({b['mis_only_false']})",
    "놓침 ↓": rate(b["miss"], b["scored"]), "음성(D · L) 오판정 ↓": f"{b['neg']}/{b['neg_n']}", "거짓_과장 후보": f"{b['false_cand']}",
    "다른 유형 위반 행에 선 거짓_과장 ↓": f"{b['false_on_other']}", "광고 문구 줄의 유형 오인 ↓": rate(b["ad_mis"], b["ad_scored"])}}
for name in RUNS:
    R, m = RESULTS[name], M[name]
    _t[name] = {
        "6종 PR-AUC 시드 평균": f"{R['prauc6_mean']:.4f} ± {R['prauc6_std']:.4f}", "선택 시드": f"{R['best_seed']} (epoch {R['seeds'][R['best_seed']]['best_epoch']})",
        "선택 시드 PR-AUC (dev 전체 · 광고 문구 줄)": f"{m['prauc6']:.3f} · {m['ad_prauc6']:.3f}",
        "위반 탐지 ↑": rate(m["det"], m["scored"]), "정답 유형 적중 ↑": rate(m["hit"], m["scored"]), "유형 오인 ↓ (거짓_과장 하나만)": f"{rate(m['mis'], m['scored'])} ({m['mis_only_false']})",
        "놓침 ↓": rate(m["miss"], m["scored"]), "음성(D · L) 오판정 ↓": f"{m['neg']}/{m['neg_n']} (D {m['D'][0]} · L {m['L'][0]})", "거짓_과장 후보": f"{m['false_cand']} (맞음 {m['false_tp']})",
        "다른 유형 위반 행에 선 거짓_과장 ↓": f"{m['false_on_other']}", "광고 문구 줄의 유형 오인 ↓": rate(m["ad_mis"], m["ad_scored"])}
print(f"■ 표 1 · dev {len(dev_rows)}행 — 가장 좋은 시드 · 그 시드의 문턱 (유형 있는 위반 {int(SC.sum())} · 음성 {int(NEG.sum())} · 광고 문구 줄의 위반 {int((SC & AD).sum())})")
display(pd.DataFrame(_t).T)
print("시드별 (자기 문턱) — 유형 오인 · 위반 탐지 · 음성 오판정 · 다른 유형 위반 행에 선 거짓_과장:")
for name in RUNS:
    print(f"  {name}: " + "  /  ".join(f"시드 {s} → {v['mis']} · {v['det']} · {v['neg']} · {v['false_on_other']}" for s, v in M[name]["per_seed"].items()))

# ── 표 2 · 유형별
_t = {}
for i, l in enumerate(CONF8):
    n = int(Y8[:, i].sum())
    _t[l] = {"dev 양성": f"{n}" + (" (측정 불가 n<30)" if n < 30 else ""),
             "Baseline 재현율": "—" if l not in b["type_tp"] else f"{b['type_tp'][l][0] / b['type_tp'][l][1] * 100:.1f}%"}
    for name in RUNS:
        s = M[name]["type"][l]
        _t[l][name] = f"R {s['tp'] / max(s['n'], 1) * 100:.1f}% · P {s['tp'] / max(s['pp'], 1) * 100:.1f}% · τ {RESULTS[name]['thresholds'][l]}"
print("\n■ 표 2 · 유형별 재현율 R · 정밀도 P · 문턱 τ")
display(pd.DataFrame(_t).T)

# ── 표 3 · 통과 기준 · 목표
VERDICT, _t = {}, {}
for name in RUNS:
    R, m = RESULTS[name], M[name]
    d_pr = R["prauc6_mean"] - b_mean
    d_rec = m["det"] / m["scored"] - b["det"] / b["scored"]
    d_type = {l: m["type"][l]["tp"] / m["type"][l]["n"] - tp / n for l, (tp, n) in b["type_tp"].items() if m["type"][l]["n"] >= 30}
    worst = min(d_type, key=d_type.get) if d_type else None
    d_neg = m["neg"] - b["neg"]
    ok = {"B": d_pr >= -GATE["prauc_drop"], "G1": d_rec >= -GATE["recall_drop"] - 1e-12,
          "G2": (d_type[worst] >= -GATE["type_recall_drop"] - 1e-12) if worst else True, "G3": d_neg <= GATE["neg_flag_up"]}
    fails = [k for k, v in ok.items() if not v]
    tg, goal = TARGET.get(name), {}
    if tg and "mis_max" in tg:
        goal = {f"유형 오인 {m['mis']}행 (목표 {tg['mis_max']} 이하)": m["mis"] <= tg["mis_max"],
                f"다른 유형 위반 행에 선 거짓_과장 {m['false_on_other']}행 (목표 {tg['false_on_other_max']} 이하)": m["false_on_other"] <= tg["false_on_other_max"]}
    elif tg:
        goal = {f"광고 문구 줄의 유형 오인 {m['ad_mis']}행 (목표 {tg['ad_mis_max']} 이하)": m["ad_mis"] <= tg["ad_mis_max"],
                f"광고 문구 줄의 6종 PR-AUC {m['ad_prauc6']:.4f} (목표 {b['ad_prauc6'] - tg['ad_prauc_drop']:.4f} 이상)": m["ad_prauc6"] >= b["ad_prauc6"] - tg["ad_prauc_drop"]}
    goal_met = bool(goal) and all(goal.values())
    if fails:
        verdict = "탈락 (" + " · ".join(fails) + ")"
    elif d_pr >= GATE["adopt_prauc"] or goal_met:
        verdict = "채택 후보"
    else:
        verdict = "통과 · 목표 미달" if goal else "통과 · 차이 없음"
    VERDICT[name] = {"verdict": verdict, "fails": fails, "goal": {k: bool(v) for k, v in goal.items()}, "d_prauc": d_pr, "d_recall": d_rec, "d_neg": d_neg,
                     "worst_type": worst, "d_worst": d_type.get(worst)}
    _t[name] = {"B · PR-AUC 시드 평균": f"{R['prauc6_mean']:.4f} ({d_pr:+.4f})", "G1 · 위반 탐지율": f"{rate(m['det'], m['scored'])} ({d_rec * 100:+.1f}%p)",
                "G2 · 유형별 최대 하락": "—" if worst is None else f"{worst} {d_type[worst] * 100:+.1f}%p",
                "G3 · 음성 오판정": f"{m['neg']}/{m['neg_n']} ({d_neg:+d}행)",
                "목표": " / ".join(("✅ " if v else "❌ ") + k for k, v in goal.items()) or "— (Baseline 재현)", "미리 본 판정": verdict}
print("\n■ 표 3 · 통과 기준 · 목표 — Baseline(v10 · 규약 값) 대비")
display(pd.DataFrame(_t).T)
print("※ 미리 보는 값이다. 판정은 공통 채점 스크립트로 하고(섹션 8), 채택 후보가 나와도 test 는 팀장 판정 뒤 1회다 (D-175).")
if "Baseline재현" in RUNS:
    _m = M["Baseline재현"]
    print(f"Baseline 재현 — 시드별 PR-AUC {[round(RESULTS['Baseline재현']['seeds'][s]['prauc6'], 4) for s in SEEDS]} (v10 {b['seed_prauc']}) · 위반 탐지 {_m['det']} (v10 {b['det']}) · 유형 오인 {_m['mis']} (v10 {b['mis']})")
if SMOKE:
    print("⚠️ SMOKE 실행 — 위 수치는 의미 없다")

SUMMARY = {"notebook": NOTEBOOK, "golden_sha256": GOLDEN_SHA, "dev_sha": DEV_SHA, "dev_rows": len(dev_rows), "gate": GATE, "target": TARGET,
           "baseline": BASELINE, "smoke": SMOKE, "train": TRAIN_STATS, "verdict": VERDICT,
           "runs": {k: {"best_seed": RESULTS[k]["best_seed"], "prauc6_mean": RESULTS[k]["prauc6_mean"], "prauc6_std": RESULTS[k]["prauc6_std"],
                        "thresholds": RESULTS[k]["thresholds"], "loss_weight_rule": RUN_DEFS[k],
                        "seeds": {str(s): {x: v[x] for x in ("prauc6", "prauc8", "best_epoch", "epochs_run", "curve", "minutes", "thresholds")}
                                  for s, v in RESULTS[k]["seeds"].items()},
                        "dev": {x: v for x, v in M[k].items() if x != "per_seed"},
                        "dev_by_seed": {str(s): v for s, v in M[k]["per_seed"].items()}} for k in RUNS}}
json.dump(SUMMARY, open(f"{OUT_ROOT}/results_v10_cards.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=float)
print("\n결과 폴더:", OUT_ROOT)
for f in sorted(os.listdir(OUT_ROOT)):
    if os.path.isfile(f"{OUT_ROOT}/{f}"):
        print(f"  {f:72s} {os.path.getsize(f'{OUT_ROOT}/{f}') / 1e6:8.1f} MB")

## 8. 다음 단계

**1) 받을 파일** — Drive `copylane_v10_cards/` 에서 실험마다 **시드별 CSV 3개 + `_scheme.zip`** 을 받아 저장소 밖(예: `Downloads\copylane_v10\`)에 둔다. 모델 zip 은 채택 후보일 때만 받는다.

**2) 공통 채점 (로컬 · 판정은 여기서 한다)** — 카드 3 의 예. 카드 4 는 이름과 `--target` 만 바꾼다.

```
python docs/psj/e2e_prototype/score_encoder_dev.py --probs <카드3_seed42.csv> <카드3_seed43.csv> <카드3_seed44.csv> --scheme <카드3_scheme.zip> --baseline-probs <v10-합성O-이유O_dev행별확률.csv> --baseline-scheme <v10-합성O-이유O_scheme.zip> --baseline-seed-prauc 0.7167 0.7081 0.7165 --target "유형 오인 60 → 46행 이하 · 다른 유형 위반 행에 선 거짓_과장 후보 87 → 60행 이하" --out-md docs/psj/reports/v10_카드3_dev채점_20261007.md
```

**3) 무엇을 보고 고르나**

- 통과 기준에 하나라도 걸리면 **탈락**이다. 다만 「문턱을 맞춘 비교」를 같이 읽어 탈락이 품질 탓인지 문턱이 옮겨 간 탓인지 적는다.
- 걸리지 않고 목표를 전부 채우면 **채택 후보** → 다른 한 사람(소성민)이 같은 설정으로 재현 → test 여부를 팀장에게 묻는다.
- 목표는 결과를 본 뒤 고치지 않는다. 가중치 3 을 다른 값으로 바꿔 보려면 새 카드를 쓴다.

**4) 올리지 않는 것** — 모델 zip · 행별 확률 CSV · 이 노트북의 실행 출력은 저장소에 올리지 않는다 (D-249). 노트북은 출력을 지우고 올린다.